In [1]:
import re
import json
from thefuzz import fuzz
from thefuzz import process as process

unique_values = json.load(open("../dependencies/unique_values_22_02_24_preprocessed.json"))
def normalize_string(s):
    # Remove special characters and convert to lower case
    return re.sub(r'\W+', '', s).lower()


In [2]:
unique_values.keys()

dict_keys(['BRAND', 'POLYMER', 'PROPERTY', 'FEATURE', 'FILLER', 'GRADE', 'CERTIFICATION', 'COMPETITOR_GRADE', 'APPLICATION', 'MODIFIER', 'UNIT', 'FILLER_PERCENTAGE', 'GRADE_WITHOUT_BRAND', 'COMPETITOR_GRADE_TRANSFORMED', 'COMP_GRADE_WITHOUT_BRAND', 'COMP_GRADE_TRANSFORMED_WITHOUT_BRAND'])

In [19]:
import re
pattern = r'\(([A-Za-z]+)\)'


In [21]:
new_applications = []
for app in unique_values['APPLICATION']:
    matches = re.findall(pattern, app)
    # if "(" in app:
    #     print(app,matches)
    new_applications+=matches
new_applications=[app.strip() for app in new_applications]
new_applications=[app for app in new_applications if len(app)>2]
new_applications = list(set(new_applications))
new_applications.append("5g")
new_applications.append("ip")

In [23]:
len(new_applications)

433

In [26]:
unique_values['APPLICATION'] = list(set(unique_values['APPLICATION'] + ["libs"] + new_applications))
with open("../dependencies/unique_values_22_02_24_preprocessed.json", "w") as fp:
    json.dump(unique_values , fp)

In [2]:
def find_substring_items(list1, list2):
    result = []
    for item2 in list2:
        for item1 in list1:
            if len(item1)>2 and item1 in item2:
                # print({item1:item2})
                result.append(item1)
                break  # Break to avoid duplicate entries of the same item2
    return result

items_to_remove_from_comp_grade = find_substring_items(unique_values['BRAND'], unique_values['COMPETITOR_GRADE'])
set(items_to_remove_from_comp_grade)

{'eco', 'forprene', 'minlon', 'talcoprene'}

In [3]:
for item_to_remove in ['forprene','minlon','talcoprene','zytel htn','zytel htn (pa)',
                       'zytel htn (pa-i)','zytel htn (pa6t/66)']:
    try: unique_values['BRAND'].remove(item_to_remove)
    except: pass


for item_to_remove in ['mechanically recycled content', 'non - chlorine & non - bromine material','non - halogenated material',
                       'non-chlorine & non-bromine material','non-halogenated material','compression molding, mold temperature',
                       'emissions','odor','resis to heat','resistance to heat','mi value','load','',
                       'non-halogenated material','non - halogenated material'],:
    try: unique_values['PROPERTY'].remove(item_to_remove)
    except: pass


for item_to_remove in ['hb','v-1', 'v-2']:
    try: unique_values['MODIFIER'].remove(item_to_remove)
    except: pass

for item_to_remove in ['0','profile','profile extrusion']:
    try: unique_values['APPLICATION'].remove(item_to_remove)
    except: pass

for item_to_remove in ['htn']:
    try: unique_values['POLYMER'].remove(item_to_remove)
    except: pass

for item_to_remove in ['cp']:
    try: unique_values['CERTIFICATION'].remove(item_to_remove)
    except: pass

In [4]:
for item_to_remove in  ["non halogenated and red phosphorus free flame resistance",
"non halogenated and red phosphorus free flame resistant",
"non halogenated and red phosphorus free flame retardant",
"non halogenated flame resistance",
"non halogenated flame resistant",
"non halogenated flame retardant",
"non red phosphorus flame retardant",
"non-halogenated/red phosphorus free flame res",
"non-halogenated/red phosphorus free flame resis",
"non-halogenated/red phosphorus free flame resist",
"non-halogenated/red phosphorus free flame resistance",
"non-halogenated/red phosphorus free flame resistant",
"non-halogenated/red phosphorus free flame resistivity",
"non-halogenated/red phosphorus free flame retardant",
"non - halogenated / red phosphorus free flame res",
"non - halogenated / red phosphorus free flame resis",
"non - halogenated / red phosphorus free flame resist",
"non - halogenated / red phosphorus free flame resistance",
"non - halogenated / red phosphorus free flame resistant",
"non - halogenated / red phosphorus free flame resistivity",
"non - halogenated / red phosphorus free flame retardant",

"red phosphorus free flame resistance",
"red phosphorus free flame resistant",
"red phosphorus free flame retardant"]:
    try: unique_values['FEATURE'].remove(item_to_remove)
    except: pass
    

In [5]:
items_to_add_cert = []
items_to_remove_cert = []
for item in unique_values['CERTIFICATION']:
    if "/" in item and ("dbl" in item or "vw" in item):
    #    print(item)
       item_new = item.replace("mercedes - benz","").replace("mercedes-benz","").replace("vw group","").replace("renault","").replace("hyundai","").replace("general motors","").replace("tesla","").strip()
       items = item_new.split("/")
       try:
        if ("dbl" in items[0] and "dbl" in items[1]) or ("vw" in items[0] and "vw" in items[1]):
            # print(items)
            items_to_remove_cert.append(item)
            items_to_add_cert += [value.strip() for value in items]
       except: pass
items_to_add_cert = list(set(items_to_add_cert))
items_to_add_cert

['dbl5406.22 pa66 gf30',
 'vw50127 pa66-7',
 'vw 50133 pa66-6-a',
 'dbl5406.21 pa66 gf30',
 'vw 50133 pa66-6 - a']

In [6]:
for item_to_remove in items_to_remove_cert:
    try: unique_values['CERTIFICATION'].remove(item_to_remove)
    except: pass


unique_values['CERTIFICATION'] = list(set(unique_values['CERTIFICATION'] + items_to_add_cert))

In [7]:
feature_variation = [item for item in unique_values['FEATURE'] if len(item.split())==2 and len(item.split()[1])>4 and not any(chr.isdigit() for chr in item)]
unique_values['FEATURE'] = list(set(unique_values['FEATURE']+ ['non red phosphorus', 'red phosphorus free','non - halogenated','non-halogenated'] +["".join(item.split()) for item in feature_variation]))

In [8]:
unique_values['FILLER'] = list(set(unique_values['FILLER'] + ['long glass fiber']))

In [9]:
unique_values['GRADE_WITHOUT_BRAND'] = list(set(unique_values['GRADE_WITHOUT_BRAND'] + ["dym", "eco-b", "eco-r", "esd", "fit", "frhr", "hfs", "hhr",
"hrlm", "hrt", "hsl", "hslr", "hte", "htn", "htr", "ice", 
"icf", "lcpa", "lds", "lof", "lof2", "med", "pcxxx", "pls/xt", 
"scxxx", "sea", "slidex", "wrf", "xap", "xap2", "xfr", "xgc"]))


unique_values['PROPERTY'] = list(set(unique_values['PROPERTY'] + ["rohs"]))
unique_values['BRAND'] = list(set(unique_values['BRAND'] + ["htn"]))

In [10]:
unique_values.keys()

dict_keys(['BRAND', 'POLYMER', 'PROPERTY', 'FEATURE', 'FILLER', 'GRADE', 'CERTIFICATION', 'COMPETITOR_GRADE', 'APPLICATION', 'MODIFIER', 'UNIT', 'FILLER_PERCENTAGE', 'GRADE_WITHOUT_BRAND', 'COMP_GRADE_WITHOUT_BRAND'])

In [11]:

for key in ['BRAND','GRADE','COMPETITOR_GRADE','GRADE_WITHOUT_BRAND', 'COMP_GRADE_WITHOUT_BRAND']:
    unique_values[key] = list(set([value.replace('™','').replace('®','') for value in unique_values[key]]))

In [12]:
unique_values['PROPERTY']

['n impact',
 'compstr1',
 'clte parallel at -40-23°c',
 'heat deflection temperature method a 1.8 mpa',
 'feucht',
 'min. melt temperature',
 'mold shrinkage normal',
 'tensile elongation at break',
 'compression set, 100°c, 70h, type 1, method b astm d395 (%)',
 'bdehn204',
 'ul recognition (1.6)',
 'flammability ( thickness 3 mm / 6 mm )',
 'tracking resist',
 'stress at break ( 160°c )',
 'charpy impact strength ( + 23°c )',
 'charpy impact strength(unnotched)',
 'temp. of deflection under load iso 75-1, -2 1.8 mpa (°c)',
 'charpy - kerbschlagzähigkeit',
 'humid',
 'dielectric strength, 23°c ( ac )',
 'hopper',
 'melt mass-flow rate temperature',
 'stressb5-40c',
 'compression set 23°c 168h type 1 method b',
 'relative permittivity iec 60250 60hz',
 'tensile module',
 'charpy notched impact strength (+23°c), 3mm, pc only',
 'flexural strain at failure astm d790 tape 0° (%)',
 'trtr p',
 'flame rating ( 1.6 mm )',
 'tensile elongation at yield, perpendicular astm d638 (%)',
 'flamma

In [15]:
for key in unique_values.keys():
    unique_values[key] = list(set(unique_values[key]))

In [17]:
with open("../dependencies/unique_values_01_02_24_preprocessed.json", "w") as fp:
    json.dump(unique_values , fp)